# Lab 2 : OD Matrices analytics

We want to find out which are more likely class of users of car sharing system. For this we compare the OD matrices obtained from different sources. 

## Setup of datasets and visualization

In [1]:
%pip install geopandas

  Using cached geopandas-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached pyogrio-0.13.0-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (5.8 kB)
  Using cached pyproj-3.8.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached shapely-2.2.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (6.7 kB)
Using cached geopandas-1.2.0-py3-none-any.whl (368 kB)
Using cached pyogrio-0.13.0-cp311-abi3-manylinux_2_28_x86_64.whl (33.3 MB)
Using cached pyproj-3.8.0-cp313-cp313-manylinux_2_28_x86_64.whl (10.0 MB)
Using cached shapely-2.2.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (2.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [geopandas]/4 [geopandas]
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install "folium>=0.12" matplotlib mapclassify

  Using cached folium-1.0.0-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached mapclassify-2.11.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached branca-0.8.2-py3-none-any.whl.metadata (1.7 kB)
Using cached folium-1.0.0-py2.py3-none-any.whl (130 kB)
Using cached mapclassify-2.11.0-py3-none-any.whl (478 kB)
Using cached branca-0.8.2-py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [folium]


In [3]:
import pymongo as pm
from pprint import pprint
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timezone
import geopandas as gpd
from shapely.geometry import box
from geopandas import GeoDataFrame
from shapely.geometry import box
from branca.colormap import linear
from IPython.display import display
import seaborn as sns
import ipywidgets as widgets

### Car sharing - MongoDB

In [4]:
client = pm.MongoClient(
    'bigdatadb.polito.it',
    ssl=True,
    authSource='carsharing',
    username='ictts',
    password='Ict4SM22!',
    tlsAllowInvalidCertificates=True
)

In [5]:
db = client['carsharing']
pb_coll = db['ictts_PermanentBookings']
pb_coll_enjoy= db['ictts_enjoy_PermanentBookings']

In [6]:
list(pb_coll.find().limit(1))

[{'_id': ObjectId('5a3bb3af20c3dd47cf59003d'),
  'init_fuel': 53,
  'vendor': 'car2go',
  'exterior': 'GOOD',
  'vin': 'WME4533421K152664',
  'final_address': 'Corso Regina Margherita, 107, 10122 Torino TO',
  'engineType': 'CE',
  'final_date': datetime.datetime(2017, 11, 1, 0, 19, 45),
  'public_transport': {'duration': -1,
   'arrival_time': -1,
   'distance': -1,
   'arrival_date': -1},
  'final_loc': {'coordinates': [7.68634, 45.07567], 'type': 'Point'},
  'smartPhoneRequired': True,
  'init_time': 1509490770,
  'interior': 'GOOD',
  'init_address': 'Lungo Dora Napoli, 18A, 10152 Torino TO',
  'init_loc': {'coordinates': [7.68486, 45.08276], 'type': 'Point'},
  'driving': {'duration': 234, 'distance': 966},
  'final_time': 1509491985,
  'plate': '288/FF214SJ',
  'city': 'Torino',
  'final_fuel': 53,
  'init_date': datetime.datetime(2017, 10, 31, 23, 59, 30),
  'walking': {'duration': 746, 'distance': 982}}]

In [7]:
list(pb_coll_enjoy.find().limit(1))

[{'_id': ObjectId('5a3bb23920c3dd4716519e40'),
  'final_fuel': 56,
  'init_fuel': 57,
  'plate': 'EZ807DD',
  'car_category_id': 8,
  'driving': {'distance': 3948, 'duration': 545},
  'onClick_disabled': False,
  'init_loc': {'coordinates': [7.7029123, 45.07156], 'type': 'Point'},
  'walking': {'distance': 3776, 'duration': 2849},
  'virtual_rental_id': 361384,
  'car_category_type_id': 1,
  'carModelData': [99, 100, 141],
  'virtual_rental_type_id': 2,
  'final_address': 'Via Arcangelo Corelli, 41, 10154 Torino TO',
  'init_address': 'Corso Carlo Luigi Farini, 20, 10153 Torino TO',
  'final_time': 1509491472,
  'public_transport': {'distance': 4034,
   'duration': 1377,
   'arrival_time': 1509493666,
   'arrival_date': datetime.datetime(2017, 11, 1, 0, 47, 46)},
  'init_date': datetime.datetime(2017, 10, 31, 23, 58, 50),
  'final_date': datetime.datetime(2017, 11, 1, 0, 11, 12),
  'city': 'Torino',
  'vendor': 'enjoy',
  'init_time': 1509490730,
  'final_loc': {'coordinates': [7.70973

In [8]:
DoWMornings=pb_coll_enjoy.aggregate([
    {'$project': {
        'hour':{'$hour':'$init_date'},
        'day':{'$dayOfWeek':'$init_date'},
        'init_loc':1,
        'final_loc':1
    }},
    {'$match':{
        'hour': {'$gte':0,'$lt':12},
        'day':{'$gte':2,'$lt':7}
    }} ,
    {'$count':'tot'}
])

In [9]:
result = next(DoWMornings, None)
print(result)

{'tot': 23148}


### UnipolData

In [10]:
df_Uni_Info = pd.read_csv("data/Unipoldata/Info_TO.csv")
print(df_Uni_Info.head())       
print(df_Uni_Info.info())  

df_Uni_Trips = pd.read_csv("data/Unipoldata/Trips_OD_TO.csv")
print(df_Uni_Trips.head())       
print(df_Uni_Trips.info())  

   id_veicolo commerciale  eta_approx genere
0           1           N          47      M
1           2           N          54      M
2           3           N          34      F
3           4           N          81      M
4           5           N          23      M
<class 'pandas.DataFrame'>
RangeIndex: 37949 entries, 0 to 37948
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id_veicolo   37949 non-null  int64
 1   commerciale  37949 non-null  str  
 2   eta_approx   37949 non-null  int64
 3   genere       37949 non-null  str  
dtypes: int64(2), str(2)
memory usage: 1.2 MB
None
   id_veicolo  id_viaggio        istante_start  lat_start  lon_start  \
0           1           2  2024-08-27 21:15:03     45.069      7.698   
1           1           4  2024-08-27 21:22:47     45.069      7.698   
2           1           5  2024-08-28 08:16:42     45.083      7.675   
3           1           6  2024-08-28 11:47:37     45.

### IMQ

## Question 1

### Import of Lab1 filtering

In [11]:
Q = 0.90

In [12]:
def load_wtime(db, coll, carsharing=True):
    if carsharing:
        pipe = [
            {
                "$match": {
                    "driving.duration": {"$gt": 0},
                    "driving.distance": {"$gt": 0}
                }
            },
            {
                "$project": {
                    "_id": 0,
                    "init_time": 1,
                    "final_time": 1,
                    "duration": "$driving.duration",
                    "day": {
                        "$dateToString": {
                            "date": "$init_date",
                            "format": "%Y-%m-%d"
                        }
                    },
                    "week": {"$isoWeek": "$init_date"},
                    "iso_year": {"$isoWeekYear": "$init_date"},
                    "dow": {"$isoDayOfWeek": "$init_date"},
                    "hour": {"$hour": "$init_date"},
                    "vendor": 1,
                    "init_address": 1,
                    "final_address": 1,
                    "plate": 1,
                    "init_loc": 1,
                    "final_loc": 1,
                    "driving": 1,
                    "walking": 1,
                    'init_date':1
                }
            }
        ]

    else:
        raise NotImplementedError(
            "Non-carsharing data processing is not implemented yet."
        )

    collection = db[coll] if isinstance(coll, str) else coll
    df = pd.DataFrame(collection.aggregate(pipe, allowDiskUse=True))

    if df.empty:
        print("Warning: MongoDB query returned no documents.")
        return df

    df["duration_min"] = df["duration"] / 60

    df["driving_distance_km"] = df["driving"].apply(
        lambda x: x["distance"] / 1000
        if isinstance(x, dict) else np.nan
    )

    df["walking_duration_min"] = df["walking"].apply(
        lambda x: x["duration"] / 60
        if isinstance(x, dict) else np.nan
    )

    return df



def best_zone(x, q=Q, log=True, threshold=False, min_duration=1.0):
    x = np.sort(np.asarray(x, dtype=float))
    if threshold:
        x = x[x >= min_duration]
    else:
        x = x[x >= 0]
    n = len(x)
    if n == 0:
        return np.nan, np.nan
    k = int(np.ceil(q * n))
    t = np.log(x) if log else x
    widths = t[k - 1:] - t[:n - k + 1]
    i = np.argmin(widths)
    return x[i], x[i + k - 1]


def keep_best_zone(df, q=Q, col="duration_min", log=True, threshold=False):
    a, b = best_zone(df[col].values, q, log, threshold)
    return df[df[col].between(a, b)].copy(), (a, b)


def get_filtered_data(db,coll,q=Q,carsharing=True,verbose=True,return_extra=False):
    raw = load_wtime(db=db,coll=coll,carsharing=carsharing)

    if raw.empty:
        raise ValueError("No data returned by MongoDB.")

    filtered, zone = keep_best_zone(raw, q=q)

    if verbose:
        a, b = zone
        n0, n1 = len(raw), len(filtered)
        percentage = n1 / n0 if n0 else 0

        print(
            f"zone = [{a:.1f}, {b:.1f}] min "
            f"-> {n1}/{n0} gardés ({percentage:.1%})"
        )

    if return_extra:
        return filtered, {"raw": raw, "zone": zone}

    return filtered

In [13]:
df_Car2Go = get_filtered_data(
    db=db,
    coll=pb_coll,
    q=0.90,
    carsharing=True
)

zone = [5.0, 22.0] min -> 35276/39190 gardés (90.0%)


In [14]:
df_Enjoy = get_filtered_data(
    db=db,
    coll=pb_coll_enjoy,
    q=0.90,
    carsharing=True
)

zone = [5.0, 22.6] min -> 32158/35726 gardés (90.0%)


### See if filtering needed

### a - Compute OD matrices on different periods

Méthode :
- Définir un zonage spatial fixe pour toute la ville.
- Associer une zone d'origine et une zone de destination à chaque trajet.
- Regrouper les trajets selon leur période temporelle.
- Calculer une matrice OD pour chaque période.

In [15]:
def prepare_od_data(df, epsg=32632):
    data = df.copy()
    # longitude and latitude
    data["origin_lon"] = data["init_loc"].apply(lambda x: x["coordinates"][0])
    data["origin_lat"] = data["init_loc"].apply(lambda x: x["coordinates"][1])

    data["destination_lon"] = data["final_loc"].apply(lambda x: x["coordinates"][0])
    data["destination_lat"] = data["final_loc"].apply(lambda x: x["coordinates"][1])

    # Project coordinates into a metric CRS
    origins = gpd.GeoSeries(gpd.points_from_xy(data["origin_lon"], data["origin_lat"]),crs="EPSG:4326").to_crs(epsg=epsg)

    destinations = gpd.GeoSeries(gpd.points_from_xy(data["destination_lon"], data["destination_lat"]),crs="EPSG:4326").to_crs(epsg=epsg)

    data["start_x"] = origins.x
    data["start_y"] = origins.y
    data["end_x"] = destinations.x
    data["end_y"] = destinations.y

    data["init_date"] = pd.to_datetime(data["init_date"])

    return data

In [16]:
def make_od_grid(data, h=1000):  #from ex2
    # Fixed grid origin
    x0 = np.floor(min(data["start_x"].min(), data["end_x"].min()) / h) * h
    y0 = np.floor(min(data["start_y"].min(), data["end_y"].min()) / h) * h
    x_max = max(data["start_x"].max(), data["end_x"].max())
    y_max = max(data["start_y"].max(), data["end_y"].max())

    n_cols = int(np.floor((x_max - x0) / h)) + 1
    n_rows = int(np.floor((y_max - y0) / h)) + 1

    polygons = []
    for row in range(n_rows):
        for col in range(n_cols):
            x = x0 + col * h
            y = y0 + row * h
            polygons.append(box(x, y, x + h, y + h))

    zones = gpd.GeoDataFrame({"zone_id": range(n_rows * n_cols)},
                             geometry=polygons,crs="EPSG:32632")

    # Assign origins and destinations
    trips = data.copy()
    
    # Remove missing or infinite coordinates
    cols = ["start_x", "start_y", "end_x", "end_y"]
    trips = trips.replace([np.inf, -np.inf], np.nan)
    trips = trips.dropna(subset=cols).copy()

    origin_col = np.floor((trips["start_x"] - x0) / h).astype(int)
    origin_row = np.floor((trips["start_y"] - y0) / h).astype(int)
    destination_col = np.floor((trips["end_x"] - x0) / h).astype(int)
    destination_row = np.floor((trips["end_y"] - y0) / h).astype(int)

    trips["origin"] = origin_row * n_cols + origin_col
    trips["destination"] = destination_row * n_cols + destination_col

    return trips, zones

#### computation of OD - trip generation

In [17]:
def compute_marginalproba(trips,zones):
    # value count compte les occurences dans chaque zone des origines et départs
    # reindex(zones.index, fill_value=0) : il garantit que les zones vides soient conservées et recoivent un 0
    zones["origin_count"] = trips["origin"].value_counts().reindex(zones.index, fill_value=0)
    zones["destination_count"] = trips["destination"].value_counts().reindex(zones.index, fill_value=0)
    zones["pO"] = zones["origin_count"] / N_train # proba marginale
    zones["pD"] = zones["destination_count"] / N_train
    
    print("Sum of pO:", zones["pO"].sum())
    print("Sum of pD:", zones["pD"].sum())
    display(zones[["zone_id", "origin_count", "destination_count", "pO", "pD"]].head())

In [18]:
def compute_ODmat(trips, zones):
    OD_counts = pd.crosstab(trips["origin"], trips["destination"])
    OD_counts = OD_counts.reindex(index=zones.index, columns=zones.index, fill_value=0)
    OD_counts.index.name = "origin"
    OD_counts.columns.name = "dest"
    
    OD_probability = OD_counts / N_train
    
    print("OD matrix shape:", OD_counts.shape)
    print("Total observed trips:", OD_counts.to_numpy().sum())
    print("Total joint probability:", OD_probability.to_numpy().sum())
    print("Symmetric count matrix:", OD_counts.equals(OD_counts.T))
    display(OD_probability.iloc[:10, :10])

In [19]:
def compute_od_by_period(trips, zones):
    trips = trips.copy()
    trips["init_date"] = pd.to_datetime(trips["init_date"])

    trips["day_type"] = np.where(
        trips["init_date"].dt.dayofweek < 5,
        "Weekdays",
        "Weekends"
    )

    trips["time_type"] = np.where(
        (trips["init_date"].dt.hour >= 6) &
        (trips["init_date"].dt.hour < 22),
        "Daytime",
        "Night-time"
    )

    od_results = {}

    categories = {
        "Weekdays": trips[trips["day_type"] == "Weekdays"],
        "Weekends": trips[trips["day_type"] == "Weekends"],
        "Daytime": trips[trips["time_type"] == "Daytime"],
        "Night-time": trips[trips["time_type"] == "Night-time"]
    }

    for name, group in categories.items():
        if group.empty:
            continue

        # OD count matrix
        OD_counts = pd.crosstab(
            group["origin"],
            group["destination"]
        ).reindex(
            index=zones.index,
            columns=zones.index,
            fill_value=0
        )

        OD_counts.index.name = "origin"
        OD_counts.columns.name = "dest"

        # Joint probability matrix
        N = len(group)
        OD_probability = OD_counts / N

        od_results[name] = {
            "counts": OD_counts,
            "probability": OD_probability,
            "n_trips": N
        }

        print(
            f"{name}: {N} trips | "
            f"Probability sum: {OD_probability.to_numpy().sum():.4f}"
        )

    return od_results

In [20]:
def plot_od_periods(OD_results, zones):

    zones_maps = {}

    for name, result in OD_results.items():

        OD = result["probability"]

        # Origin and destination probabilities
        pO = OD.sum(axis=1)
        pD = OD.sum(axis=0)

        gdf = zones.copy()

        gdf["pO"] = pO.reindex(zones.index).values
        gdf["pD"] = pD.reindex(zones.index).values

        gdf["origin_count"] = result["counts"].sum(axis=1).values
        gdf["destination_count"] = result["counts"].sum(axis=0).values

        zones_maps[name] = gdf

    # Common scales
    vmax = max(
        max(gdf["pO"].max(), gdf["pD"].max())
        for gdf in zones_maps.values()
    )

    # Create maps
    outputs = []

    for name, gdf in zones_maps.items():

        out = widgets.Output(
            layout=widgets.Layout(width="50%")
        )

        with out:

            print(name)

            m = gdf.explore(
                column="pO",
                cmap=linear.YlOrRd_09.scale(0, vmax),
                tiles="OpenStreetMap",
                tooltip=[
                    "zone_id",
                    "origin_count",
                    "destination_count",
                    "pO",
                    "pD"
                ],
                style_kwds={
                    "fillOpacity": 0.5,
                    "weight": 0.5
                },
                legend=True
            )

            display(m)

        outputs.append(out)

    display(
        widgets.VBox([
            widgets.HBox(outputs[:2]),
            widgets.HBox(outputs[2:])
        ])
    )

    return zones_maps

In [21]:
def compare_od_percentages(OD_results):
    comparisons = {
        "Weekdays vs Weekends": ("Weekdays", "Weekends"),
        "Daytime vs Night-time": ("Daytime", "Night-time")
    }

    results = {}

    for title, (a, b) in comparisons.items():
        P = OD_results[a]["probability"].to_numpy()
        Q = OD_results[b]["probability"].to_numpy()

        # Total variation distance
        difference = 0.5 * np.abs(P - Q).sum() * 100
        results[title] = difference
    return pd.Series(results, name="OD difference (%)")

For Car2Go

In [22]:
trips_car2Go = prepare_od_data(df_Car2Go)
tripsC2G,zonesC2G=make_od_grid(trips_car2Go)

In [23]:
N_train = len(tripsC2G)
compute_marginalproba(tripsC2G, zonesC2G)

Sum of pO: 1.0
Sum of pD: 1.0


,zone_id,origin_count,destination_count,pO,pD
0,0,5,14,0.000157,0.000441
1,1,68,113,0.002141,0.003557
2,2,83,109,0.002613,0.003431
3,3,103,60,0.003242,0.001889
4,4,244,144,0.007681,0.004533


In [24]:
compute_ODmat(tripsC2G, zonesC2G)

OD matrix shape: (120, 120)
Total observed trips: 31767
Total joint probability: 1.0
Symmetric count matrix: False


dest,0,1,2,3,4,5,6,7,8,9
dest,,,,,,,,,,
0,0.000000,0.000000,0.000031,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.000000,0.000000,0.000031,0.000031,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.000000,0.000000,0.000063,0.0,0.0,0.0,0.0,0.0
3,0.000031,0.000252,0.000063,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
4,0.000031,0.000031,0.000315,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
7,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


In [25]:
compute_marginalproba(tripsC2G, zonesC2G)

Sum of pO: 1.0
Sum of pD: 1.0


,zone_id,origin_count,destination_count,pO,pD
dest,,,,,
0,0,5,14,0.000157,0.000441
1,1,68,113,0.002141,0.003557
2,2,83,109,0.002613,0.003431
3,3,103,60,0.003242,0.001889
4,4,244,144,0.007681,0.004533


In [26]:
od_by_period=compute_od_by_period(tripsC2G, zonesC2G)

OD_weekdays = od_by_period["Weekdays"]["probability"]
OD_weekends = od_by_period["Weekends"]["probability"]

OD_daytime = od_by_period["Daytime"]["probability"]
OD_night = od_by_period["Night-time"]["probability"]

Weekdays: 23055 trips | Probability sum: 1.0000
Weekends: 8712 trips | Probability sum: 1.0000
Daytime: 22780 trips | Probability sum: 1.0000
Night-time: 8987 trips | Probability sum: 1.0000


In [27]:
zones_maps = plot_od_periods(od_by_period, zonesC2G)
differences = compare_od_percentages(od_by_period)
display(differences.round(2))

Weekdays vs Weekends     26.22
Daytime vs Night-time    25.42
Name: OD difference (%), dtype: float64

#### compare side length

side_lengths = sorted(set(list(range(300, 3001, 300)) + [1000]))
results = []

for h in side_lengths:
    trips, zones = make_od_grid(tripsC2G, h)
    K = len(zones)
    active_zones = pd.concat([trips["origin"], trips["destination"]]).nunique()
    observed_pairs = len(trips[["origin", "destination"]].drop_duplicates())
    intrazonal = (trips["origin"] == trips["destination"]).sum()

    results.append({"side_m": h, "K": K, 
                    "active_training_zones": active_zones,
                    "k^2": K**2,
                    "observed_training_OD": observed_pairs,
                    "mean_trips_per_OD": len(trips) / (K * K),
                    "intrazonal_count": intrazonal,
                    "intrazonal_percent": 100 * intrazonal / len(trips)})

zoning_results = pd.DataFrame(results)
zoning_results.round(4)


1000x1000 justification:
- 120 zones K^2 = 120^2 = 14,400 theoritical OD pairs (dim ok for analysis of matrices)
- good cover of displacements : 3 709 distincts OD pairs, against only 203 3,000 m grid. --> good spatial diversity
- not much intrazone displacements: 0,76 % of trips
- trade off between resolution and sparsity

for h in [300, 1000, 3000]:
    trips, zones = make_od_grid(trips, h)
    print("Square side:", h, "m | Number of zones:", len(zones))
    display(zones.explore(tiles="OpenStreetMap", color="blue", tooltip="zone_id",
                          style_kwds={"fillOpacity": 0.03, "weight": 0.7}))

### b - OD matrices on different dates

In [28]:
def compute_od_by_period_diffdates(trips, zones, weeks, start_hour=6, end_hour=22):

    trips = trips.copy()
    trips["init_date"] = pd.to_datetime(trips["init_date"])

    # Filter out weekends (keep Monday to Friday)
    trips = trips[trips["init_date"].dt.dayofweek < 5].copy()

    # Extract ISO week and year
    iso = trips["init_date"].dt.isocalendar()
    trips["week"] = iso.week.astype(int)

    trips["iso_year"] = iso.year.astype(int)

    od_results = {}

    for week in weeks:

        # Same period: weekdays and same hours
        group = trips[
            (trips["week"] == week) &
            (trips["iso_year"] == 2017) &
            (trips["init_date"].dt.dayofweek < 5) &
            (trips["init_date"].dt.hour >= start_hour) &
            (trips["init_date"].dt.hour < end_hour)
        ].copy()

        if group.empty:
            print(f"Week {week}: no trips found")
            continue

        # OD count matrix using existing zone IDs
        OD_counts = pd.crosstab(
            group["origin"],
            group["destination"]
        ).reindex(
            index=zones["zone_id"],
            columns=zones["zone_id"],
            fill_value=0
        )

        OD_counts.index.name = "origin"
        OD_counts.columns.name = "destination"

        # Joint probability matrix
        N = len(group)
        OD_probability = OD_counts / N

        name = f"Week {week}"

        od_results[name] = {
            "counts": OD_counts,
            "probability": OD_probability,
            "n_trips": N
        }

        print(
            f"{name}: {N} trips | "
            f"Probability sum: {OD_probability.to_numpy().sum():.4f}"
        )

    return od_results


In [29]:
def plot_od_periods_diffdates(OD_results, zones):
    zones_maps = {}

    # Prepare maps for each week
    for name, result in OD_results.items():

        OD = result["probability"]
        # Marginal probabilities
        pO = OD.sum(axis=1)
        pD = OD.sum(axis=0)

        gdf = zones.copy()
        gdf["pO"] = pO.reindex(zones.index).fillna(0).values
        gdf["pD"] = pD.reindex(zones.index).fillna(0).values

        gdf["origin_count"] = (result["counts"].sum(axis=1).reindex(zones.index).fillna(0).values)

        gdf["destination_count"] = (result["counts"].sum(axis=0).reindex(zones.index).fillna(0).values)

        zones_maps[name] = gdf

    # Common scale for all maps
    vmax = max(
        max(gdf["pO"].max(), gdf["pD"].max())
        for gdf in zones_maps.values()
    )

    # Create plots
    outputs = []

    for name, gdf in zones_maps.items():

        for column, title in [
            ("pO", "Origin probability"),
            ("pD", "Destination probability")
        ]:

            out = widgets.Output(
                layout=widgets.Layout(width="50%")
            )

            with out:

                print(f"{name} - {title}")

                m = gdf.explore(
                    column=column,
                    cmap=linear.YlOrRd_09.scale(0, vmax),
                    tiles="OpenStreetMap",
                    tooltip=[
                        "zone_id",
                        "origin_count",
                        "destination_count",
                        "pO",
                        "pD"
                    ],
                    style_kwds={
                        "fillOpacity": 0.5,
                        "weight": 0.5
                    },
                    legend=True
                )

                display(m)

            outputs.append(out)

    # Display maps two by two
    rows = []

    for i in range(0, len(outputs), 2):
        rows.append(widgets.HBox(outputs[i:i+2]))

    display(widgets.VBox(rows))

    return zones_maps


In [30]:
def compare_od_percentages_diffdates(OD_results):

    weeks = list(OD_results.keys())

    results = {}

    # Compare each pair of weeks
    for i in range(len(weeks)):
        for j in range(i + 1, len(weeks)):

            week_a = weeks[i]
            week_b = weeks[j]

            P = OD_results[week_a]["probability"].to_numpy()
            Q = OD_results[week_b]["probability"].to_numpy()

            # Total variation distance (%)
            difference = 0.5 * np.abs(P - Q).sum() * 100

            title = f"{week_a} vs {week_b}"

            results[title] = difference

    return pd.Series(results, name="OD difference (%)")


In [31]:
trips_car2Go = prepare_od_data(df_Car2Go)
tripsC2G,zonesC2G=make_od_grid(trips_car2Go,h=1000)
N_train = len(tripsC2G)
compute_marginalproba(tripsC2G, zonesC2G)

OD_results = compute_od_by_period_diffdates(
    tripsC2G, zonesC2G,
    weeks=[40, 41], # we have 35 to 44
    start_hour=8,
    end_hour=10
)

zones_maps = plot_od_periods_diffdates(OD_results, zonesC2G)

compare_od_percentages_diffdates(OD_results)

Sum of pO: 1.0
Sum of pD: 1.0


,zone_id,origin_count,destination_count,pO,pD
0,0,5,14,0.000157,0.000441
1,1,68,113,0.002141,0.003557
2,2,83,109,0.002613,0.003431
3,3,103,60,0.003242,0.001889
4,4,244,144,0.007681,0.004533


Week 40: 971 trips | Probability sum: 1.0000
Week 41: 1273 trips | Probability sum: 1.0000


Week 40 vs Week 41    64.32327
Name: OD difference (%), dtype: float64

### c - compute OD matrices on same date for Car2GO and Enjoy

In [32]:
def compute_od_by_day(trips, zones, selected_date):

    trips = trips.copy()
    trips["init_date"] = pd.to_datetime(trips["init_date"])

    # Filter trips for the selected day
    selected_date = pd.to_datetime(selected_date).date()

    trips = trips[
        trips["init_date"].dt.date == selected_date
    ].copy()

    if trips.empty:
        print(f"No trips found for {selected_date}")
        return {}

    # Classify trips by time
    trips["time_type"] = np.where(
        (trips["init_date"].dt.hour >= 6) &
        (trips["init_date"].dt.hour < 22),
        "Daytime",
        "Night-time"
    )

    od_results = {}

    categories = {
        "All day": trips,
        "Daytime": trips[trips["time_type"] == "Daytime"],
        "Night-time": trips[trips["time_type"] == "Night-time"]
    }

    for name, group in categories.items():

        if group.empty:
            continue

        OD_counts = pd.crosstab(
            group["origin"],
            group["destination"]
        ).reindex(
            index=zones.index,
            columns=zones.index,
            fill_value=0
        )

        OD_counts.index.name = "origin"
        OD_counts.columns.name = "dest"

        N = len(group)
        OD_probability = OD_counts / N

        od_results[name] = {
            "counts": OD_counts,
            "probability": OD_probability,
            "n_trips": N
        }

        print(
            f"{name}: {N} trips | "
            f"Probability sum: {OD_probability.to_numpy().sum():.4f}"
        )

    return od_results

In [33]:

def plot_od_day(OD_results, zones_dict):

    zones_maps = {}

    # Prepare maps for each operator and period
    for operator, periods in OD_results.items():

        zones = zones_dict[operator]

        for period, result in periods.items():

            OD = result["probability"]

            pO = OD.sum(axis=1)
            pD = OD.sum(axis=0)

            gdf = zones.copy()

            gdf["pO"] = pO.reindex(zones.index).fillna(0).values
            gdf["pD"] = pD.reindex(zones.index).fillna(0).values

            gdf["origin_count"] = (
                result["counts"].sum(axis=1)
                .reindex(zones.index).fillna(0).values
            )

            gdf["destination_count"] = (
                result["counts"].sum(axis=0)
                .reindex(zones.index).fillna(0).values
            )

            zones_maps[(operator, period)] = gdf

    # Common scale across operators and periods
    vmax = max(
        max(gdf["pO"].max(), gdf["pD"].max())
        for gdf in zones_maps.values()
    )

    outputs = []

    for (operator, period), gdf in zones_maps.items():

        for column, title in [
            ("pO", "Origin probability"),
            ("pD", "Destination probability")
        ]:

            out = widgets.Output(
                layout=widgets.Layout(width="50%")
            )

            with out:

                print(f"{operator} - {period} - {title}")

                m = gdf.explore(
                    column=column,
                    cmap=linear.YlOrRd_09.scale(0, vmax),
                    tiles="OpenStreetMap",
                    tooltip=[
                        "zone_id",
                        "origin_count",
                        "destination_count",
                        "pO",
                        "pD"
                    ],
                    style_kwds={
                        "fillOpacity": 0.5,
                        "weight": 0.5
                    },
                    legend=True
                )

                display(m)

            outputs.append(out)

    rows = []

    for i in range(0, len(outputs), 2):
        rows.append(widgets.HBox(outputs[i:i+2]))

    display(widgets.VBox(rows))

    return zones_maps


In [34]:
def start_end_collection(coll):
    init = coll.find({},{'init_time':1,'_id':0}).sort("init_time", 1).limit(1)
    end = coll.find({},{'final_time':1,'_id':0}).sort("init_time", -1).limit(1)
    init = next(init)['init_time']
    end = next(end)['final_time']
    return init, end

In [35]:
start_end_collection(pb_coll)

(1504216803, 1509491985)

In [36]:
start_end_collection(pb_coll_enjoy)

(1504217009, 1509491472)

2017-09-01 → 2017-10-31

In [37]:
trips_car2Go = prepare_od_data(df_Car2Go)
tripsC2G,zonesC2G=make_od_grid(trips_car2Go,h=1000)
N_train = len(tripsC2G)
compute_marginalproba(tripsC2G, zonesC2G)

trips_Enjoy = prepare_od_data(df_Enjoy)
tripsEnjoy,zonesEnjoy=make_od_grid(trips_Enjoy,h=1000)
N_train = len(tripsEnjoy)
compute_marginalproba(tripsEnjoy, zonesEnjoy)

OD_results = {
    "Car2Go": compute_od_by_day(
        tripsC2G, zonesC2G,
        selected_date="2017-09-25"
    ),
    "Enjoy": compute_od_by_day(
        tripsEnjoy, zonesEnjoy,
        selected_date="2017-09-25"
    )
}

zones_dict = {
    "Car2Go": zonesC2G,
    "Enjoy": zonesEnjoy
}

zones_maps = plot_od_day(OD_results, zones_dict)

Sum of pO: 1.0
Sum of pD: 1.0


,zone_id,origin_count,destination_count,pO,pD
0,0,5,14,0.000157,0.000441
1,1,68,113,0.002141,0.003557
2,2,83,109,0.002613,0.003431
3,3,103,60,0.003242,0.001889
4,4,244,144,0.007681,0.004533


Sum of pO: 1.0
Sum of pD: 1.0


,zone_id,origin_count,destination_count,pO,pD
0,0,0,0,0.000000,0.000000
1,1,0,0,0.000000,0.000000
2,2,0,0,0.000000,0.000000
3,3,48,24,0.001657,0.000828
4,4,33,29,0.001139,0.001001


All day: 159 trips | Probability sum: 1.0000
Daytime: 56 trips | Probability sum: 1.0000
Night-time: 103 trips | Probability sum: 1.0000
All day: 665 trips | Probability sum: 1.0000
Daytime: 524 trips | Probability sum: 1.0000
Night-time: 141 trips | Probability sum: 1.0000


## Question 2

Goal :
- Does 2 OD mat of my dataset are similars? 
- How does metrics react when we compare two random matrices or utterly different ones ?
- Which is the most pertinent metric to compare my OD? 

We already have normalized mat.
We can test:
- L1
- L2
- Linf
- Spectral
- Jensen Shanon ?

In [43]:
def compute_od_metrics(A, B):
    """
    Compute several distance metrics between two OD matrices
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    if not np.isfinite(A).all() or not np.isfinite(B).all():
        raise ValueError("Matrices contain NaN or infinite values.")

    if (A < 0).any() or (B < 0).any():
        raise ValueError("OD matrices must be non-negative.")

    elif A.sum() <= 0 or B.sum() <= 0:
        raise ValueError("Matrices must have positive sums.")

    diff = A - B

    # L1
    d1 = np.abs(diff).sum()

    # L2
    d2 = np.linalg.norm(diff, ord="fro")

    # L-infinity
    d_inf = np.abs(diff).max()

    # Spectral norm
    d_spectral = np.linalg.norm(diff, ord=2)

    # Jensen-Shannon distance
    p = A.flatten()
    q = B.flatten()

    m = (p + q) / 2

    def kl_divergence(x, y):
        mask = x > 0
        return np.sum(x[mask] * np.log2(x[mask] / y[mask]))

    js_divergence = (kl_divergence(p, m) + kl_divergence(q, m))/2
    js_distance = np.sqrt(js_divergence)

    return {
        "L1": d1,
        "L2_Frobenius": d2,
        "L_inf": d_inf,
        "Spectral": d_spectral,
        "Jensen_Shannon": js_distance
    }


In [44]:
# Test 1: Two similar OD matrices

A_similar = np.array([
    [0.40, 0.10],
    [0.20, 0.30]
])

B_similar = np.array([
    [0.35, 0.15],
    [0.25, 0.25]
])


# Test 2: Two random probability matrices

rng = np.random.default_rng(42)

A_random = rng.random((4, 4))
A_random = A_random / A_random.sum()

B_random = rng.random((4, 4))
B_random = B_random / B_random.sum()


# Test 3: Completely dissimilar matrices

A_dissimilar = np.array([
    [1.0, 0.0],
    [0.0, 0.0]
])

B_dissimilar = np.array([
    [0.0, 0.0],
    [0.0, 1.0]
])


# Compute all metrics

tests = {
    "Similar matrices": (A_similar, B_similar),
    "Random matrices": (A_random, B_random),
    "Completely dissimilar": (A_dissimilar, B_dissimilar)
}

results = {}

for test, (A, B) in tests.items():
    results[test] = compute_od_metrics(A, B)

results_df = pd.DataFrame(results).T

print(results_df.round(4))

                           L1  L2_Frobenius   L_inf  Spectral  Jensen_Shannon
Similar matrices       0.2000        0.1000  0.0500    0.1000          0.0921
Random matrices        0.6298        0.1953  0.0938    0.1597          0.3852
Completely dissimilar  2.0000        1.4142  1.0000    1.0000          1.0000


In [45]:
# Test on different day type for Car2Go

od_by_period=compute_od_by_period(tripsC2G, zonesC2G)
OD_weekdays = od_by_period["Weekdays"]["probability"]
OD_weekends = od_by_period["Weekends"]["probability"]
OD_daytime = od_by_period["Daytime"]["probability"]
OD_night = od_by_period["Night-time"]["probability"]



tests_car2Go = {
    "Weekdays vs Weekends matrices": (OD_weekdays, OD_weekends),
    "Daytime vs night matrices": (OD_daytime, OD_night)
}

results = {}

for test, (A, B) in tests_car2Go.items():
    results[test] = compute_od_metrics(A, B)

results_df = pd.DataFrame(results).T

print(results_df.round(4))

Weekdays: 23055 trips | Probability sum: 1.0000
Weekends: 8712 trips | Probability sum: 1.0000
Daytime: 22780 trips | Probability sum: 1.0000
Night-time: 8987 trips | Probability sum: 1.0000
                                   L1  L2_Frobenius   L_inf  Spectral  \
Weekdays vs Weekends matrices  0.5245        0.0128  0.0017    0.0048   
Daytime vs night matrices      0.5084        0.0123  0.0019    0.0042   

                               Jensen_Shannon  
Weekdays vs Weekends matrices          0.3526  
Daytime vs night matrices              0.3511  


In [57]:
from itertools import combinations
# Comparison between same day but diff dates
OD_results = compute_od_by_period_diffdates(
    tripsC2G, zonesC2G,
    weeks=[37,38,39,40], # we have 35 to 44
    start_hour=8,
    end_hour=18
)

for week_a, week_b in combinations(OD_results.keys(), 2):
    A = OD_results[week_a]["probability"]
    B = OD_results[week_b]["probability"]
    print(f"OD differences: {week_a} vs {week_b}")
    display(compute_od_metrics(A,B))

Week 37: 2183 trips | Probability sum: 1.0000
Week 38: 1815 trips | Probability sum: 1.0000
Week 39: 1213 trips | Probability sum: 1.0000
Week 40: 1612 trips | Probability sum: 1.0000
OD differences: Week 37 vs Week 38


{'L1': np.float64(1.0686590218177277),
 'L2_Frobenius': np.float64(0.03278477605462633),
 'L_inf': np.float64(0.0073482924022215245),
 'Spectral': np.float64(0.01193739928006569),
 'Jensen_Shannon': np.float64(0.6445916109391496)}

OD differences: Week 37 vs Week 39


{'L1': np.float64(1.2119688260367623),
 'L2_Frobenius': np.float64(0.03683944189065774),
 'L_inf': np.float64(0.0060453651633944235),
 'Spectral': np.float64(0.013138301471416853),
 'Jensen_Shannon': np.float64(0.7020250528760335)}

OD differences: Week 37 vs Week 40


{'L1': np.float64(1.138099617049863),
 'L2_Frobenius': np.float64(0.03338230225930182),
 'L_inf': np.float64(0.004829218333865683),
 'Spectral': np.float64(0.011474868537487682),
 'Jensen_Shannon': np.float64(0.6774114947574015)}

... (2 outputs left out)

OD differences: Week 38 vs Week 40


{'L1': np.float64(1.1268550608726562),
 'L2_Frobenius': np.float64(0.03502705926511148),
 'L_inf': np.float64(0.00702376802083547),
 'Spectral': np.float64(0.013679208615036412),
 'Jensen_Shannon': np.float64(0.6732779316400128)}

OD differences: Week 39 vs Week 40


{'L1': np.float64(1.2296287734816578),
 'L2_Frobenius': np.float64(0.03821397578519426),
 'L_inf': np.float64(0.005795364117838388),
 'Spectral': np.float64(0.013488006080206484),
 'Jensen_Shannon': np.float64(0.7088276219758977)}

In [59]:
# Prepare OD data for both operators --> need same grid
trips_car2Go = prepare_od_data(df_Car2Go)
trips_Enjoy = prepare_od_data(df_Enjoy)

# Add an operator identifier to each dataset
trips_car2Go["operator"] = "Car2Go"
trips_Enjoy["operator"] = "Enjoy"

# Combine trips and create a common spatial grid
all_trips = pd.concat([trips_car2Go, trips_Enjoy],ignore_index=True)

# Create a single spatial grid for both operators
all_trips, common_zones = make_od_grid(all_trips,h=1000)

# Separate trips by operator
tripsC2G = all_trips[all_trips["operator"] == "Car2Go"].copy()
tripsEnjoy = all_trips[all_trips["operator"] == "Enjoy"].copy()

# Compute OD matrices for the same date
selected_date = "2017-09-25"

OD_results = {
    "Car2Go": compute_od_by_day(
        tripsC2G,
        common_zones,
        selected_date=selected_date
    ),

    "Enjoy": compute_od_by_day(
        tripsEnjoy,
        common_zones,
        selected_date=selected_date
    )
}

# Compare OD matrices for each time period

comparisons = {}

periods = ["All day", "Daytime", "Night-time"]

for period in periods:

    # Retrieve probability matrices
    A = OD_results["Car2Go"][period]["probability"]
    B = OD_results["Enjoy"][period]["probability"]

    # Compute comparison metrics
    metrics = compute_od_metrics(A, B)

    comparisons[period] = metrics

    print(f"\nCar2Go vs Enjoy - {period}")
    print(pd.Series(metrics).round(5))


All day: 159 trips | Probability sum: 1.0000
Daytime: 56 trips | Probability sum: 1.0000
Night-time: 103 trips | Probability sum: 1.0000
All day: 665 trips | Probability sum: 1.0000
Daytime: 524 trips | Probability sum: 1.0000
Night-time: 141 trips | Probability sum: 1.0000

Car2Go vs Enjoy - All day
L1                1.79850
L2_Frobenius      0.09435
L_inf             0.02844
Spectral          0.03602
Jensen_Shannon    0.91282
dtype: float64

Car2Go vs Enjoy - Daytime
L1                1.92748
L2_Frobenius      0.14541
L_inf             0.03571
Spectral          0.05342
Jensen_Shannon    0.95817
dtype: float64

Car2Go vs Enjoy - Night-time
L1                1.86711
L2_Frobenius      0.13540
L_inf             0.02913
Spectral          0.05080
Jensen_Shannon    0.95476
dtype: float64
